# 固有表現認識の基礎（ガゼッティア・BIO タグ・素性テンプレートと CRF・サブワード整合）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpbook/blob/main/notebooks/ch3_ner_gazetteer_subword.ipynb)

書籍 第3章「系列ラベリング問題の解き方」（3-1 固有表現認識：3-1-1 ルールベース、3-1-2 機械学習ベース、3-1-3 前段、練習問題 1・2）に対応する演習ノートブックです。
CPU のみ・十数秒で実行でき、固有表現認識の古典的手法から深層学習の前処理・後処理までを一貫して体験します。

1. **ガゼッティア（固有表現辞書）と最長一致・多義語誤マッチ対策（ハイブリッド判定）**（3-1-1、p.85）
2. **BIO (IOB2) タグによるスパン表現と `SudachiPy` + `sklearn-crfsuite` による素性テンプレート比較実験・遷移重み分析**（3-1-2、図 3.2、練習問題 1・2）
3. **`SentencePiece` サブワード分割に対する BIO ラベル整合（`-100` マスク）と不正 `I-XXX` 遷移の後処理補正**（3-1-3 前段、p.87〜88）

In [1]:
!pip install -q sudachipy sudachidict_core sentencepiece sklearn-crfsuite

In [2]:
import tempfile
from pathlib import Path

import pandas as pd
import sentencepiece as spm
import sklearn_crfsuite
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn_crfsuite import metrics as crf_metrics
from sudachipy import dictionary, tokenizer

---

## 1. ガゼッティア（最長一致）と多義語による誤認識（3-1-1、p.85）

地名辞典・人名・組織名リスト（**ガゼッティア**）を用いた辞書マッチングでは、短い単語（「東京」「日本」）が長い固有名詞（「東京大学」「日本銀行」）の一部を誤って抜かないよう**最長一致**を使うのが基本です。
しかしそれでも、「青森（地名 vs りんご品種）」「千葉（地名 vs 苗字）」「アップル（IT企業 vs 果物）」のような**多義語**は文字列一致だけでは誤認識してしまいます。

In [3]:
gazetteer = {
    "東京": "LOC",
    "東京大学": "ORG",
    "日本": "LOC",
    "日本橋": "LOC",
    "日本銀行": "ORG",
    "青森": "LOC",
    "千葉": "LOC",
    "アップル": "ORG",
    "鈴木": "PER",
}

def extract_longest_matches(text, dic):
    results = []
    i = 0
    # 長い見出し語から優先して照合する
    keys = sorted(dic.keys(), key=len, reverse=True)
    while i < len(text):
        matched = None
        for k in keys:
            if text.startswith(k, i):
                matched = k
                break
        if matched:
            results.append((i, i + len(matched), matched, dic[matched]))
            i += len(matched)
        else:
            i += 1
    return results

sentences = [
    "東京大学の鈴木教授が日本銀行を訪れた。",
    "青森は甘みと酸味のバランスが良いりんごです。",
    "千葉さんがスーパーでアップルパイを買った。",
]

for s in sentences:
    print("文:", s)
    print("  最長一致結果:", [(m, tag) for _, _, m, tag in extract_longest_matches(s, gazetteer)])

文: 東京大学の鈴木教授が日本銀行を訪れた。
  最長一致結果: [('東京大学', 'ORG'), ('鈴木', 'PER'), ('日本銀行', 'ORG')]
文: 青森は甘みと酸味のバランスが良いりんごです。
  最長一致結果: [('青森', 'LOC')]
文: 千葉さんがスーパーでアップルパイを買った。
  最長一致結果: [('千葉', 'LOC'), ('アップル', 'ORG')]


### ハイブリッドアプローチ：辞書候補 ＋ 文脈による 2 値分類（3-1-1、p.85）

書籍 p.85 では、ノイジーなテキストに対して「**まずガゼッティアで候補文字列を取り出し、その前後の文脈から実際にその固有表現か否かを機械学習分類器で判定する**」ハイブリッド手法が紹介されています。簡易な文脈分類器で先ほどの誤マッチを抑制できるか試しましょう。

In [4]:
# (文, 注目スパンが辞書タグ通りの固有表現か: 1=はい, 0=いいえ)
train_cands = [
    ("青森県でリンゴを収穫した。", "青森", 1),
    ("夏休みに青森へ旅行した。", "青森", 1),
    ("青森出身の友人と会った。", "青森", 1),
    ("この品種の青森はとても甘い。", "青森", 0),
    ("青森とふじを食べ比べた。", "青森", 0),
    ("千葉県でイベントが開催された。", "千葉", 1),
    ("千葉駅から電車に乗った。", "千葉", 1),
    ("来週は千葉に出張する。", "千葉", 1),
    ("千葉さんが資料を作成した。", "千葉", 0),
    ("千葉部長に確認をお願いした。", "千葉", 0),
    ("アップルが新型端末を発表した。", "アップル", 1),
    ("アップル社の株価が上昇した。", "アップル", 1),
    ("アップルに転職した。", "アップル", 1),
    ("甘いアップルパイを焼いた。", "アップル", 0),
    ("アップルティーを注文した。", "アップル", 0),
]

def span_context_feature(text, start, end):
    left = text[max(0, start - 4):start]
    right = text[end:min(len(text), end + 5)]
    surf = text[start:end]
    return f"L:{left} M:{surf} R:{right}"

train_feats = []
train_y = []
for text, target, label in train_cands:
    idx = text.index(target)
    train_feats.append(span_context_feature(text, idx, idx + len(target)))
    train_y.append(label)

vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 3))
X_tr = vec.fit_transform(train_feats)
clf = LogisticRegression(C=3.0, max_iter=1000).fit(X_tr, train_y)

test_sentences = [
    "千葉に出張して青森県産のお土産を買った。",
    "千葉さんが会議室でアップルパイを食べた。",
    "アップルが開発した新しいOSが公開された。",
]

for s in test_sentences:
    raw = extract_longest_matches(s, gazetteer)
    filtered = []
    for st, ed, surf, tag in raw:
        feat = span_context_feature(s, st, ed)
        prob = clf.predict_proba(vec.transform([feat]))[0, 1]
        if prob >= 0.5:
            filtered.append((surf, tag, round(float(prob), 2)))
    print("文:", s)
    print("  辞書のみ    :", [(surf, tag) for _, _, surf, tag in raw])
    print("  ハイブリッド:", filtered)

文: 千葉に出張して青森県産のお土産を買った。
  辞書のみ    : [('千葉', 'LOC'), ('青森', 'LOC')]
  ハイブリッド: [('千葉', 'LOC', 0.68), ('青森', 'LOC', 0.69)]
文: 千葉さんが会議室でアップルパイを食べた。
  辞書のみ    : [('千葉', 'LOC'), ('アップル', 'ORG')]
  ハイブリッド: []
文: アップルが開発した新しいOSが公開された。
  辞書のみ    : [('アップル', 'ORG')]
  ハイブリッド: [('アップル', 'ORG', 0.69)]


「千葉**に**出張」「アップル**が**開発」は採用され、「千葉**さん**」「アップル**パイ**」のような別義表現を文脈から棄却できていることがわかります。

---

## 2. BIO タグによるスパン表現と素性テンプレート・CRF 実験（3-1-2、練習問題 1・2）

### 2-1. BIO (IOB2) タグによる固有表現のスパン表現（3-1、図 3.2）

固有表現認識 (NER) は、テキスト中の人名 (`PER`)・組織名 (`ORG`)・地名 (`LOC`) などのスパンを **BIO (IOB2) タグ** による系列ラベリング問題として定式化します。

- `B-X`: 固有表現 `X` の先頭 (Begin)
- `I-X`: 固有表現 `X` の内部 (Inside)
- `O`: 固有表現以外 (Outside)

`B-` と `I-` を区別することで、「東京 (`B-LOC`)」「大阪 (`B-LOC`)」のように同種の固有表現が連続しても 2 つの別々の固有表現として切り出すことができます。

In [5]:
def decode_bio(tokens, tags, return_spans=False):
    '''トークン列と BIO タグ列から固有表現スパンを抽出する'''
    entities = []
    cur_type = None
    cur_tokens = []
    start_idx = None
    for i, (tok, tag) in enumerate(zip(tokens, tags)):
        if tag.startswith("B-"):
            if cur_tokens:
                ent = (
                    (cur_type, "".join(cur_tokens), start_idx, i)
                    if return_spans
                    else (cur_type, "".join(cur_tokens))
                )
                entities.append(ent)
            cur_type = tag[2:]
            cur_tokens = [tok]
            start_idx = i
        elif tag.startswith("I-") and cur_type == tag[2:]:
            cur_tokens.append(tok)
        else:
            if cur_tokens:
                ent = (
                    (cur_type, "".join(cur_tokens), start_idx, i)
                    if return_spans
                    else (cur_type, "".join(cur_tokens))
                )
                entities.append(ent)
                cur_type = None
                cur_tokens = []
                start_idx = None
    if cur_tokens:
        ent = (
            (cur_type, "".join(cur_tokens), start_idx, len(tokens))
            if return_spans
            else (cur_type, "".join(cur_tokens))
        )
        entities.append(ent)
    return entities

sample_toks = ["山田", "さん", "は", "東京", "大阪", "間", "を", "移動", "し", "た"]
sample_tags = ["B-PER", "O", "O", "B-LOC", "B-LOC", "O", "O", "O", "O", "O"]
print("抽出された固有表現:", decode_bio(sample_toks, sample_tags))

抽出された固有表現: [('PER', '山田'), ('LOC', '東京'), ('LOC', '大阪')]


### 2-2. 素性テンプレートと CRF による比較実験（3-1-2、練習問題 2）

書籍 第3章（3-1-2・3-2、図 3.3）および練習問題 2 で解説されているとおり、CRF などの系列ラベリングモデルでは**素性テンプレート**を用いて各トークンの周辺の表層形・品詞・文字種・バイグラム素性などを動的に展開します。

ここでは `SudachiPy`（`Dictionary().create()`、Mode A）で形態素解析した小さな固有表現データセットを用意し、素性テンプレートを段階的に増やしたときの性能変化を確認します。

In [6]:
sudachi = dictionary.Dictionary().create()

# (文, [(表層文字列, ラベル), ...]) の形式で簡易データセットを定義
raw_train = [
    ("佐藤さんは東京都立大学で研究している。", [("佐藤", "PER"), ("東京都立大学", "ORG")]),
    ("鈴木さんは京都大学を卒業した。", [("鈴木", "PER"), ("京都大学", "ORG")]),
    ("高橋さんは大阪大学に着任した。", [("高橋", "PER"), ("大阪大学", "ORG")]),
    ("田中さんは名古屋大学で講義を行った。", [("田中", "PER"), ("名古屋大学", "ORG")]),
    ("伊藤さんは九州大学から東京へ出張した。", [("伊藤", "PER"), ("九州大学", "ORG"), ("東京", "LOC")]),
    ("渡辺さんは北海道大学を訪問した。", [("渡辺", "PER"), ("北海道大学", "ORG")]),
    ("山本さんは東北大学で学会に参加した。", [("山本", "PER"), ("東北大学", "ORG")]),
    ("中村さんは横浜国立大学に在籍している。", [("中村", "PER"), ("横浜国立大学", "ORG")]),
    ("小林さんは筑波大学と理化学研究所と共同研究する。", [("小林", "PER"), ("筑波大学", "ORG"), ("理化学研究所", "ORG")]),
    ("加藤さんは渋谷と新宿の間を歩いた。", [("加藤", "PER"), ("渋谷", "LOC"), ("新宿", "LOC")]),
    ("吉田さんは札幌から福岡へ移動した。", [("吉田", "PER"), ("札幌", "LOC"), ("福岡", "LOC")]),
    ("山田さんは仙台で会議に出席した。", [("山田", "PER"), ("仙台", "LOC")]),
    ("松本さんは広島と名古屋を視察した。", [("松本", "PER"), ("広島", "LOC"), ("名古屋", "LOC")]),
    ("井上さんは沖縄から東京に戻った。", [("井上", "PER"), ("沖縄", "LOC"), ("東京", "LOC")]),
    ("木村さんは神戸大学で講演した。", [("木村", "PER"), ("神戸大学", "ORG")]),
    ("林さんは一橋大学の研究会に参加した。", [("林", "PER"), ("一橋大学", "ORG")]),
]

# 訓練データには一度も登場しない人名・組織名・地名を含むテストデータ
raw_test = [
    ("斎藤さんは千葉大学で発表を行った。", [("斎藤", "PER"), ("千葉大学", "ORG")]),
    ("清水さんは金沢大学から京都へ移動した。", [("清水", "PER"), ("金沢大学", "ORG"), ("京都", "LOC")]),
    ("森さんは産業技術総合研究所を訪問した。", [("森", "PER"), ("産業技術総合研究所", "ORG")]),
    ("池田さんは長崎から横浜へ出張した。", [("池田", "PER"), ("長崎", "LOC"), ("横浜", "LOC")]),
    ("橋本さんは岡山大学で学会に参加した。", [("橋本", "PER"), ("岡山大学", "ORG")]),
    ("石川さんは熊本と鹿児島を視察した。", [("石川", "PER"), ("熊本", "LOC"), ("鹿児島", "LOC")]),
]

def align_bio(text, entities):
    '''SudachiPy のトークン列に文字オフセット基準で BIO タグを付与する'''
    spans = []
    for ent_text, ent_type in entities:
        start = text.find(ent_text)
        assert start != -1, f"{ent_text} not in {text}"
        spans.append((start, start + len(ent_text), ent_type))

    morphemes = list(sudachi.tokenize(text, tokenizer.Tokenizer.SplitMode.A))
    tokens = []
    tags = []
    for m in morphemes:
        b, e = m.begin(), m.end()
        tag = "O"
        for sb, se, etype in spans:
            if b >= sb and e <= se:
                tag = f"B-{etype}" if b == sb else f"I-{etype}"
                break
        tokens.append((m.surface(), m.part_of_speech()[0], m.part_of_speech()[1]))
        tags.append(tag)
    return tokens, tags

train_data = [align_bio(t, ents) for t, ents in raw_train]
test_data = [align_bio(t, ents) for t, ents in raw_test]

print("トークン化・BIO付与の例:", list(zip([x[0] for x in train_data[0][0]], train_data[0][1])))

トークン化・BIO付与の例: [('佐藤', 'B-PER'), ('さん', 'O'), ('は', 'O'), ('東京', 'B-ORG'), ('都立', 'I-ORG'), ('大学', 'I-ORG'), ('で', 'O'), ('研究', 'O'), ('し', 'O'), ('て', 'O'), ('いる', 'O'), ('。', 'O')]


In [7]:
def char_type(s):
    if s.isdigit():
        return "DIGIT"
    if all("ァ" <= ch <= "ヶ" or ch == "ー" for ch in s):
        return "KATA"
    if all("ぁ" <= ch <= "ん" for ch in s):
        return "HIRA"
    if all("\u4e00" <= ch <= "\u9fff" for ch in s):
        return "KANJI"
    return "MIXED"

def extract_features(sent, i, level):
    word, pos1, pos2 = sent[i]
    feats = {"bias": 1.0, "w": word}

    if level >= 2:
        feats.update({
            "pos1": pos1,
            "pos2": pos2,
            "ctype": char_type(word),
        })

    if level >= 3:
        if i > 0:
            pw, ppos1, _ = sent[i - 1]
            feats.update({
                "-1:w": pw,
                "-1:pos1": ppos1,
                "-1_0:w": f"{pw}/{word}",      # バイグラム素性
            })
        else:
            feats["BOS"] = True

        if i < len(sent) - 1:
            nw, npos1, _ = sent[i + 1]
            feats.update({
                "+1:w": nw,
                "+1:pos1": npos1,
                "0_+1:w": f"{word}/{nw}",      # バイグラム素性
            })
        else:
            feats["EOS"] = True

    return feats

def evaluate_entity_spans(dataset, y_pred):
    '''decode_bio(return_spans=True) を用いてエンティティ単位 Precision / Recall / F1 を計算する'''
    tp = fp = fn = 0
    for (sent, gold), pred in zip(dataset, y_pred):
        toks = [w for w, _, _ in sent]
        gold_spans = set(decode_bio(toks, gold, return_spans=True))
        pred_spans = set(decode_bio(toks, pred, return_spans=True))
        tp += len(gold_spans & pred_spans)
        fp += len(pred_spans - gold_spans)
        fn += len(gold_spans - pred_spans)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

results = []
models = {}
for level, desc in [
    (1, "1. 表層形のみ"),
    (2, "2. +品詞・文字種"),
    (3, "3. +前後文脈・バイグラム素性"),
]:
    X_train = [[extract_features(s, i, level) for i in range(len(s))] for s, _ in train_data]
    y_train = [tags for _, tags in train_data]
    X_test = [[extract_features(s, i, level) for i in range(len(s))] for s, _ in test_data]
    y_test = [tags for _, tags in test_data]

    crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)
    crf.fit(X_train, y_train)
    y_pred = crf.predict(X_test)
    models[level] = (crf, y_pred)

    ent_labels = ["B-PER", "B-ORG", "I-ORG", "B-LOC"]
    f1_macro = crf_metrics.flat_f1_score(y_test, y_pred, average="macro", labels=ent_labels, zero_division=0)
    acc = crf_metrics.flat_accuracy_score(y_test, y_pred)
    span_p, span_r, span_f1 = evaluate_entity_spans(test_data, y_pred)
    results.append({
        "素性テンプレート": desc,
        "トークン正解率": round(acc, 3),
        "BIO タグ単位 Macro-F1（O を除く）": round(f1_macro, 3),
        "エンティティ単位 Precision": round(span_p, 3),
        "エンティティ単位 Recall": round(span_r, 3),
        "エンティティ単位 F1（スパン完全一致）": round(span_f1, 3),
    })

pd.DataFrame(results)

,素性テンプレート,トークン正解率,BIO タグ単位 Macro-F1（O を除く）,エンティティ単位 Precision,エンティティ単位 Recall,エンティティ単位 F1（スパン完全一致）
0,1. 表層形のみ,0.857,0.614,1.0,0.6,0.75
1,2. +品詞・文字種,1.000,1.000,1.0,1.0,1.00
2,3. +前後文脈・バイグラム素性,1.000,1.000,1.0,1.0,1.00


表の各指標には次のような違いがあります。

- **トークン正解率**: テストデータ全体（70 トークン）のうち約 68.6%（48 トークン）が固有表現以外の `O` タグであるため、「1. 表層形のみ」のように未知の固有表現（15 件中 6 件、とくに地名 `LOC` は 0 件抽出）を見逃していても `0.857` と見かけ上高い値になります。
- **BIO タグ単位 Macro-F1（`O` を除く）**: `B-PER` や `I-ORG` など `O` を除いた各 BIO タグごとの F1 の平均であり、`O` の多さに引きずられず各タグの識別力を測れます。
- **エンティティ単位 Precision / Recall / F1（スパン完全一致）**: `decode_bio` で復元した固有表現スパン（種別・表層・開始位置・終了位置）が完全に一致したときのみ正解とみなす指標で、[BERT による固有表現認識](https://colab.research.google.com/github/mamorlis/nlpbook/blob/main/notebooks/ch3_bert_ner.ipynb) (`seqeval`) の評価方法と直接対応します。「1. 表層形のみ」では既知語尾（「〜大学」）を手がかりに出力できたスパン自体は合っているため Precision は `1.000` ですが、未知の地名や長い組織名を取りこぼすため Recall = `0.600`・エンティティ単位 F1 = `0.750` にとどまります。

未知の人名（斎藤・清水・森…）や地名（長崎・横浜…）は「表層形のみ」では見つけられませんが、**品詞（固有名詞）や文字種・前後の文脈（「〜さんは」「〜から」「〜へ」）の素性テンプレート**を加えることで、未知の固有表現も正しく抽出できるようになります（レベル 2・3 では全指標で `1.000`）。

最後に、素性レベル 3 の予測例と、CRF が学習したタグ間の遷移重み（高い遷移・低い遷移）を確認してみましょう。

In [8]:
crf3, y_pred3 = models[3]
for (sent, gold), pred in zip(test_data, y_pred3):
    toks = [w for w, _, _ in sent]
    print("文:", "".join(toks))
    print("  正解:", decode_bio(toks, gold))
    print("  予測:", decode_bio(toks, pred))

print("\n学習された遷移重み（上位5件・下位5件）:")
trans = sorted(crf3.transition_features_.items(), key=lambda x: x[1], reverse=True)
for (src, dst), w in trans[:5] + trans[-5:]:
    print(f"  {src:6s} -> {dst:6s}: {w:+.3f}")

文: 斎藤さんは千葉大学で発表を行った。
  正解: [('PER', '斎藤'), ('ORG', '千葉大学')]
  予測: [('PER', '斎藤'), ('ORG', '千葉大学')]
文: 清水さんは金沢大学から京都へ移動した。
  正解: [('PER', '清水'), ('ORG', '金沢大学'), ('LOC', '京都')]
  予測: [('PER', '清水'), ('ORG', '金沢大学'), ('LOC', '京都')]
文: 森さんは産業技術総合研究所を訪問した。
  正解: [('PER', '森'), ('ORG', '産業技術総合研究所')]
  予測: [('PER', '森'), ('ORG', '産業技術総合研究所')]
文: 池田さんは長崎から横浜へ出張した。
  正解: [('PER', '池田'), ('LOC', '長崎'), ('LOC', '横浜')]
  予測: [('PER', '池田'), ('LOC', '長崎'), ('LOC', '横浜')]
文: 橋本さんは岡山大学で学会に参加した。
  正解: [('PER', '橋本'), ('ORG', '岡山大学')]
  予測: [('PER', '橋本'), ('ORG', '岡山大学')]
文: 石川さんは熊本と鹿児島を視察した。
  正解: [('PER', '石川'), ('LOC', '熊本'), ('LOC', '鹿児島')]
  予測: [('PER', '石川'), ('LOC', '熊本'), ('LOC', '鹿児島')]

学習された遷移重み（上位5件・下位5件）:
  B-ORG  -> I-ORG : +2.033
  O      -> O     : +1.913
  I-ORG  -> I-ORG : +0.923
  B-PER  -> O     : +0.727
  O      -> B-LOC : +0.477
  B-PER  -> I-ORG : -0.095
  B-LOC  -> I-ORG : -0.637
  O      -> B-PER : -0.845
  B-ORG  -> O     : -1.141
  O      -> I-ORG : -1.362


---

## 3. `SentencePiece` サブワード分割と BIO タグ整合・不正遷移の補正（3-1-3、p.88）

BERT などの事前学習モデルで固有表現認識を解く際は、単語単位のアノテーション（BIO タグ）とモデル内部の**サブワード分割**を整合させる前処理・後処理が不可欠です。

1. **アラインメント（訓練・入力時）**: 1 単語が複数のサブワードに分割されたとき、**先頭サブワードだけに元の BIO タグを割り当て、2 番目以降のサブワードには損失計算から除外する `-100` を割り当てる**
2. **不正遷移の補正（推論・後処理時、3-1-3 p.88「I ラベルは単独では出現しない」）**: 文頭、`O` の直後、異なる種別のタグの直後に出た `I-X` を `B-X` に補正する

In [9]:
# 小規模コーパスで SentencePiece（サブワード分割器）を学習
corpus_lines = [
    "東京大学の鈴木先生が学会で講演した。",
    "日本銀行の本店は東京都中央区日本橋にある。",
    "2月29日にロンドンから空港を経由して帰国する。",
    "ロンドン市内のホテルから空港へ向かい帰国する。",
    "ロンドンから空港まではバスで移動する。",
    "ロンドンでは国際会議に出席して翌日に帰国する。",
    "空港の国際線ロビーで友人と合流した。",
    "羽田空港から成田空港までバスで移動した。",
    "今日は2月15日で、来週にロンドンへ出発する。",
    "ホテルから空港まで電車で移動する。",
    "コーヒーとチーズケーキをカフェで注文する。",
    "スポーツとニュースのテレビ番組を毎日見る。",
]
tmpdir = Path(tempfile.mkdtemp())
corpus_file = tmpdir / "corpus.txt"
corpus_file.write_text("\n".join(corpus_lines), encoding="utf-8")
spm.SentencePieceTrainer.Train(
    input=str(corpus_file),
    model_prefix=str(tmpdir / "sp"),
    vocab_size=115,
    model_type="unigram",
    character_coverage=1.0,
    add_dummy_prefix=False,
    minloglevel=2,
)
sp = spm.SentencePieceProcessor(model_file=str(tmpdir / "sp.model"))

# 単語単位の文と BIO ラベル（3-1 図 3.2 の例文）
words = ["2", "月", "29", "日", "に", "ロンドン", "ヒースロー", "空港", "から", "帰国", "する", "。"]
word_tags = ["B-TIME", "I-TIME", "I-TIME", "I-TIME", "O", "B-LOC", "B-LOC", "I-LOC", "O", "O", "O", "O"]

subwords = []
aligned_labels = []
word_ids = []

for w_idx, (w, tag) in enumerate(zip(words, word_tags)):
    pieces = sp.encode_as_pieces(w)
    for p_idx, p in enumerate(pieces):
        subwords.append(p)
        word_ids.append(w_idx)
        # 先頭サブワードだけタグを残し、後続サブワードは損失計算を無視 (-100) にする
        aligned_labels.append(tag if p_idx == 0 else "-100")

display(pd.DataFrame({
    "word_id": word_ids,
    "元単語": [words[i] for i in word_ids],
    "サブワード": subwords,
    "整合後ラベル": aligned_labels,
}))

,word_id,元単語,サブワード,整合後ラベル
0,0,2,2,B-TIME
1,1,月,月,I-TIME
2,2,29,2,I-TIME
3,2,29,9,-100
4,3,日,日,I-TIME
5,4,に,に,O
6,5,ロンドン,ロンドン,B-LOC
7,6,ヒースロー,ヒ,B-LOC
8,6,ヒースロー,ー,-100
9,6,ヒースロー,ス,-100


In [10]:
def repair_bio_sequence(tags):
    """書籍 3-1-3 (p.88): I ラベルは単独では出現しないため、先頭や O の直後の I-XXX を B-XXX へ補正する"""
    repaired = []
    prev_type = None
    for tag in tags:
        if tag.startswith("I-"):
            ent_type = tag[2:]
            if prev_type != ent_type:
                repaired.append(f"B-{ent_type}")
                prev_type = ent_type
                continue
        if tag.startswith("B-"):
            prev_type = tag[2:]
        elif tag == "O":
            prev_type = None
        else:
            prev_type = tag[2:]
        repaired.append(tag)
    return repaired

# 例えばヒースロー(B-LOC) が O に誤予測され、空港(I-LOC) だけ残ったケース
noisy_pred = ["O", "O", "O", "O", "O", "B-LOC", "O", "I-LOC", "O", "O", "O", "O"]
fixed_pred = repair_bio_sequence(noisy_pred)

print("補正前の単語ラベル:", list(zip(words[5:9], noisy_pred[5:9])))
print("補正後の単語ラベル:", list(zip(words[5:9], fixed_pred[5:9])))

補正前の単語ラベル: [('ロンドン', 'B-LOC'), ('ヒースロー', 'O'), ('空港', 'I-LOC'), ('から', 'O')]
補正後の単語ラベル: [('ロンドン', 'B-LOC'), ('ヒースロー', 'O'), ('空港', 'B-LOC'), ('から', 'O')]


次のステップとして、日本語 BERT (`tohoku-nlp/bert-base-japanese-v3`) と `-100` マスク整合・`seqeval` によるエンティティ単位評価を用いたファインチューニングおよび未知固有表現 (OOV) への汎化検証は、[BERT による固有表現認識 (Token Classification)](https://colab.research.google.com/github/mamorlis/nlpbook/blob/main/notebooks/ch3_bert_ner.ipynb) を参照してください。